## MHW detection: percentile thresholds

Implementing the Hobday et al. (2016) methodology for marine heatwave detection. Starting with a small preview: the 90th percentile SST for a single calendar day, across all available years, before generalizing to every day of the year.

June 15 is used here rather than a peak-summer day, since it falls before the June 21 cutoff, so it has 11 years of coverage, unlike days after June 21, which have only 10. (see year coverage note in notebook 02).

In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

In [3]:
# 90th percentile SST for June 15, across all 11 years in the dataset
result = con.execute("""
    SELECT PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY analysed_sst) - 273.15 AS p90_celsius,
           COUNT(*) AS n_observations
    FROM '../data/processed/med_sst_2016_2026.parquet'
    WHERE MONTH(time) = 6 AND DAY(time) = 15
""").df()

print(result)

   p90_celsius  n_observations
0    24.469993          131296


**Results**: 24.47 C, calculated from 131,296 observations (11 years x ~11,936 sea cells per day). As expected, this is well above the June average of 22.2 C established in notebook 02: the threshold is meant to mark the boundary between normal and exceptional, not represent a typical day.

In [4]:
# 90th percentile SST across all available years (varies by calendar day, see note below)
result = con.execute("""
    SELECT MONTH(time) AS month, DAY(time) AS day, PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY analysed_sst) - 273.15 AS p90_celsius,
           COUNT(*) AS n_observations
    FROM '../data/processed/med_sst_2016_2026.parquet'
    GROUP BY month, day
    ORDER BY month, day
""").df()

print(result)

     month  day  p90_celsius  n_observations
0        1    1    16.239994          131296
1        1    2    16.159994          131296
2        1    3    16.059994          131296
3        1    4    16.009994          131296
4        1    5    15.939994          131296
..     ...  ...          ...             ...
361     12   27    16.509994          119360
362     12   28    16.419994          119360
363     12   29    16.329994          119360
364     12   30    16.339994          119360
365     12   31    16.289994          119360

[366 rows x 4 columns]


In [5]:
# checking the result against the 15th June 90th percentile SST
print(result[(result["month"] == 6) & (result["day"] == 15)])

     month  day  p90_celsius  n_observations
166      6   15    24.469993          131296


**Results**: 24.469993 C, matching exactly the isolated single-day calculation from earlier in the notebook. Confirms the grouped query (366 daily thresholds) produces the same result as filtering for one specific day manually, no discrepancy introduced by the GROUP BY.

In [6]:
# Temporarily raise the row display limit to inspect all 366 days at once
with pd.option_context("display.max_rows", None):
    print(result)

     month  day  p90_celsius  n_observations
0        1    1    16.239994          131296
1        1    2    16.159994          131296
2        1    3    16.059994          131296
3        1    4    16.009994          131296
4        1    5    15.939994          131296
5        1    6    15.849994          131296
6        1    7    15.839994          131296
7        1    8    15.809994          131296
8        1    9    15.709994          131296
9        1   10    15.659994          131296
10       1   11    15.589994          131296
11       1   12    15.529994          131296
12       1   13    15.459994          131296
13       1   14    15.399994          131296
14       1   15    15.319994          131296
15       1   16    15.179994          131296
16       1   17    15.119994          131296
17       1   18    15.149994          131296
18       1   19    15.079994          131296
19       1   20    14.989994          131296
20       1   21    14.989994          131296
21       1

**Results**: 366 daily thresholds calculated, one per calendar day. Verified against the isolated June 15 calculation from earlier in the notebook (24.47 C in both cases), confirming the grouped query produces consistent results. Observation counts vary by exact calendar day, not by month: days from January 1 through June 21 have 131,296 observations (11 years of coverage), while June 22 through December 31 have 119,360 (10 years): the cutoff falls mid-June because the dataset ends 2026-06-21, not at a month boundary.

## Consecutive day detection: gaps and islands

The Hobday et al. definition requires at least 5 consecutive days above the percentile threshold to count as an event, not just individual days that happen to exceed it. This needs a way to count "how many consecutive days above threshold, up to and including today" for every row.

Before applying this to the real dataset, I test the logic on a small fake sequence, using the gaps-and-islands SQL pattern: two progressive row counters (one over all rows, one within each state group) whose difference stays constant during an unbroken run and changes at every interruption.

In [7]:
# Test the gaps-and-islands logic on a small fake sequence before applying it to real data
test_query = """
    SELECT * FROM (
        VALUES 
            (1, 'N'), (2, 'N'), (3, 'Y'), (4, 'Y'), (5, 'Y'), 
            (6, 'Y'), (7, 'Y'), (8, 'N'), (9, 'N'), (10, 'Y'), (11, 'Y')
    ) AS t(day, state)
"""
result_test = con.execute(test_query).df()
print(result_test)

    day state
0     1     N
1     2     N
2     3     Y
3     4     Y
4     5     Y
5     6     Y
6     7     Y
7     8     N
8     9     N
9    10     Y
10   11     Y


In [8]:
# Add the two progressive counters: one over all rows, one within each state group
test_query_2 = """
    SELECT 
        day,
        state,
        ROW_NUMBER() OVER (ORDER BY day) AS row_num,
        ROW_NUMBER() OVER (PARTITION BY state ORDER BY day) AS state_num,
        ROW_NUMBER() OVER (ORDER BY day) - ROW_NUMBER() OVER (PARTITION BY state ORDER BY day) AS group_id
    FROM (
        VALUES 
            (1, 'N'), (2, 'N'), (3, 'Y'), (4, 'Y'), (5, 'Y'), 
            (6, 'Y'), (7, 'Y'), (8, 'N'), (9, 'N'), (10, 'Y'), (11, 'Y')
    ) AS t(day, state)
    ORDER BY day
"""
result_test_2 = con.execute(test_query_2).df()
print(result_test_2)

    day state  row_num  state_num  group_id
0     1     N        1          1         0
1     2     N        2          2         0
2     3     Y        3          1         2
3     4     Y        4          2         2
4     5     Y        5          3         2
5     6     Y        6          4         2
6     7     Y        7          5         2
7     8     N        8          3         5
8     9     N        9          4         5
9    10     Y       10          6         4
10   11     Y       11          7         4


**Results**: the pattern works as expected. Days 3-7 (five consecutive Y) all share `group_id = 2`, correctly identifying them as one block. Days 10-11 (two consecutive Y) share `group_id = 4`, a separate, shorter block, correctly distinguished from the first. The logic is confirmed on fake data.

**Next step**: apply it to the real dataset, joining the raw SST observations against the daily percentile thresholds calculated earlier, to build the actual "above threshold" flag for every row.

In [9]:
# Save the daily percentile thresholds as a Parquet file, so DuckDB can join against them
result.to_parquet("../data/processed/daily_thresholds.parquet")

In [10]:
# Sanity check: confirm the thresholds table has all 366 calendar days, none missing
check_query = """
    SELECT COUNT(*) AS n_threshold_days
    FROM '../data/processed/daily_thresholds.parquet'
"""
print(con.execute(check_query).df())

   n_threshold_days
0               366


In [11]:
# Join raw observations against daily thresholds, flag each row as above/below threshold
join_query = """
    SELECT 
        raw.time,
        raw.latitude,
        raw.longitude,
        raw.analysed_sst,
        thresh.p90_celsius,
        CASE WHEN (raw.analysed_sst - 273.15) > thresh.p90_celsius THEN 'Y' ELSE 'N' END AS above_threshold
    FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
    INNER JOIN '../data/processed/daily_thresholds.parquet' AS thresh
        ON MONTH(raw.time) = thresh.month AND DAY(raw.time) = thresh.day
    ORDER BY raw.time, raw.latitude, raw.longitude
"""

result_join = con.execute(join_query).df()
print(result_join.shape)
print(result_join.head())

(45655200, 6)
        time   latitude  longitude  analysed_sst  p90_celsius above_threshold
0 2016-01-01  40.507652   2.043520    288.579994    16.239994               N
1 2016-01-01  40.507652   2.093567    288.649994    16.239994               N
2 2016-01-01  40.507652   2.143612    288.699994    16.239994               N
3 2016-01-01  40.507652   2.193659    288.739994    16.239994               N
4 2016-01-01  40.507652   2.243704    288.749994    16.239994               N


## Applying gaps-and-islands to real data

Testing the consecutive-day logic on a single grid cell first (latitude 42.664433, longitude 10.401196), before scaling to the full bounding box (next session).

In [12]:
# Pick a single cell (one lat/lon pair) and look at its full time series with the Y/N flag
single_cell_query = """
    SELECT 
        raw.time,
        raw.latitude,
        raw.longitude,
        raw.analysed_sst - 273.15 AS sst_celsius,
        thresh.p90_celsius,
        CASE WHEN (raw.analysed_sst - 273.15) > thresh.p90_celsius THEN 'Y' ELSE 'N' END AS above_threshold
    FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
    INNER JOIN '../data/processed/daily_thresholds.parquet' AS thresh
        ON MONTH(raw.time) = thresh.month AND DAY(raw.time) = thresh.day
    WHERE raw.latitude = 42.664433 AND raw.longitude = 10.401196
    ORDER BY raw.time
"""
result_single_cell = con.execute(single_cell_query).df()
print(result_single_cell.shape)
print(result_single_cell.head())

(3825, 6)
        time   latitude  longitude  sst_celsius  p90_celsius above_threshold
0 2016-01-01  42.664433  10.401196    15.839994    16.239994               N
1 2016-01-02  42.664433  10.401196    15.659994    16.159994               N
2 2016-01-03  42.664433  10.401196    15.649994    16.059994               N
3 2016-01-04  42.664433  10.401196    15.659994    16.009994               N
4 2016-01-05  42.664433  10.401196    15.659994    15.939994               N


In [13]:
# Apply gaps-and-islands logic to this single cell's time series
gaps_islands_query = """
    WITH flagged AS (
        SELECT 
            raw.time,
            CASE WHEN (raw.analysed_sst - 273.15) > thresh.p90_celsius THEN 'Y' ELSE 'N' END AS above_threshold
        FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
        INNER JOIN '../data/processed/daily_thresholds.parquet' AS thresh
            ON MONTH(raw.time) = thresh.month AND DAY(raw.time) = thresh.day
        WHERE raw.latitude = 42.664433 AND raw.longitude = 10.401196
    )
    SELECT 
        time,
        above_threshold,
        ROW_NUMBER() OVER (ORDER BY time) 
            - ROW_NUMBER() OVER (PARTITION BY above_threshold ORDER BY time) AS group_id
    FROM flagged
    ORDER BY time
"""
result_gaps = con.execute(gaps_islands_query).df()
print(result_gaps.shape)

(3825, 3)


In [14]:
print(result_gaps[result_gaps["above_threshold"] == "Y"].head(15))

          time above_threshold  group_id
193 2016-07-12               Y       193
348 2016-12-14               Y       347
349 2016-12-15               Y       347
445 2017-03-21               Y       442
451 2017-03-27               Y       447
467 2017-04-12               Y       462
468 2017-04-13               Y       462
469 2017-04-14               Y       462
470 2017-04-15               Y       462
471 2017-04-16               Y       462
472 2017-04-17               Y       462
473 2017-04-18               Y       462
520 2017-06-04               Y       508
521 2017-06-05               Y       508
532 2017-06-16               Y       518


**Results**: 3825 rows, one `group_id` label per day for this single cell. Spot-checking Y-flagged rows confirms the logic works correctly on real data: April 12-18, 2017 (`group_id` 462) is a genuine 7-day consecutive run, already exceeding the 5-day minimum for an event. June 4-5, 2017 (`group_id` 508) is only 2 consecutive days, correctly falling short of the threshold. Interestingly, December 14-15, 2016 also appears as a short 2-day above-threshold run (group_id 347), consistent with the Mistral-driven variability discussed in notebook 02. Logic confirmed via a CTE on real data.

**Next step**: extend PARTITION BY to include latitude and longitude, so the consecutive-day count runs independently for every grid cell, not just this one anchor point.

## Scaling gaps-and-islands to the full bounding box

Extending the consecutive-day logic from the single test cell to every grid cell in the dataset, by adding latitude and longitude to both ROW_NUMBER() partitions. This means the consecutive-day count restarts independently for each grid cell, rather than mixing sequences from different locations.

In [15]:
full_gaps_islands_query = """
    WITH flagged AS (
        SELECT 
            raw.time, raw.latitude, raw.longitude,
            CASE WHEN (raw.analysed_sst - 273.15) > thresh.p90_celsius THEN 'Y' ELSE 'N' END AS above_threshold
        FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
        INNER JOIN '../data/processed/daily_thresholds.parquet' AS thresh
            ON MONTH(raw.time) = thresh.month AND DAY(raw.time) = thresh.day
    )
    SELECT 
        time, latitude, longitude,
        above_threshold,
        ROW_NUMBER() OVER (PARTITION BY latitude, longitude ORDER BY time) 
            - ROW_NUMBER() OVER (PARTITION BY latitude, longitude, above_threshold ORDER BY time) AS group_id
    FROM flagged
    ORDER BY time
"""
result_gaps_full = con.execute(full_gaps_islands_query).df()
print(result_gaps_full.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(45655200, 5)


## Extracting valid marine heatwave events

With the consecutive-day count now calculated per cell (group_id), the final step filters for genuine events: consecutive runs of at least 5 days above threshold, per Hobday et al. Using HAVING to filter groups after aggregation (COUNT(*) >= 5), since the row count only exists once rows are grouped.

In [16]:
events_query = """
    SELECT 
        latitude, 
        longitude, 
        group_id,
        MIN(time) AS event_start,
        MAX(time) AS event_end,
        COUNT(*) AS duration_days
    FROM (SELECT * FROM result_gaps_full)
    WHERE above_threshold = 'Y'
    GROUP BY latitude, longitude, group_id
    HAVING COUNT(*) >= 5
    ORDER BY event_start
"""
result_events = con.execute(events_query).df()
print(result_events.shape)
print(result_events.head())

(253145, 6)
    latitude  longitude  group_id event_start  event_end  duration_days
0  41.159702  11.602299         0  2016-01-01 2016-01-14             14
1  40.557808   6.097242         0  2016-01-01 2016-01-17             17
2  40.708282  11.952620         0  2016-01-01 2016-01-15             15
3  40.507652   6.497610         0  2016-01-01 2016-01-17             17
4  40.708282  12.102757         0  2016-01-01 2016-01-06              6


In [17]:
# Sanity check: verify these flagged rows are genuinely 'Y', not a filtering bug
check = result_gaps_full[
    (result_gaps_full["latitude"] == 40.708282) & 
    (result_gaps_full["longitude"] == 5.446646) &
    (result_gaps_full["time"] >= "2016-01-01") &
    (result_gaps_full["time"] <= "2016-02-15")
]
print(check[["time", "above_threshold", "group_id"]])

Empty DataFrame
Columns: [time, above_threshold, group_id]
Index: []


In [18]:
# Retry with a small tolerance for floating point precision
check = result_gaps_full[
    (result_gaps_full["latitude"].round(4) == round(40.708282, 4)) & 
    (result_gaps_full["longitude"].round(4) == round(5.446646, 4)) &
    (result_gaps_full["time"] >= "2016-01-01") &
    (result_gaps_full["time"] <= "2016-02-15")
]
print(check[["time", "above_threshold", "group_id"]])

             time above_threshold  group_id
2282   2016-01-01               Y         0
14215  2016-01-02               Y         0
26150  2016-01-03               Y         0
37764  2016-01-04               Y         0
49709  2016-01-05               Y         0
61658  2016-01-06               Y         0
73593  2016-01-07               Y         0
85847  2016-01-08               Y         0
97775  2016-01-09               Y         0
109704 2016-01-10               Y         0
121633 2016-01-11               Y         0
133604 2016-01-12               Y         0
145545 2016-01-13               Y         0
157440 2016-01-14               Y         0
169397 2016-01-15               Y         0
181364 2016-01-16               Y         0
192954 2016-01-17               Y         0
204898 2016-01-18               Y         0
216820 2016-01-19               Y         0
228756 2016-01-20               Y         0
241032 2016-01-21               Y         0
252971 2016-01-22               

In [19]:
# Compare actual SST vs threshold for this suspicious cell, January 2016
check_temps = con.execute("""
    SELECT raw.time, raw.analysed_sst - 273.15 AS sst_celsius, thresh.p90_celsius
    FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
    INNER JOIN '../data/processed/daily_thresholds.parquet' AS thresh
        ON MONTH(raw.time) = thresh.month AND DAY(raw.time) = thresh.day
    WHERE raw.latitude = 40.708282 AND raw.longitude = 5.446646
        AND raw.time BETWEEN '2016-01-01' AND '2016-02-15'
    ORDER BY raw.time
""").df()
print(check_temps)

Empty DataFrame
Columns: [time, sst_celsius, p90_celsius]
Index: []


In [20]:
check_temps = con.execute("""
    SELECT raw.time, raw.analysed_sst - 273.15 AS sst_celsius, thresh.p90_celsius
    FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
    INNER JOIN '../data/processed/daily_thresholds.parquet' AS thresh
        ON MONTH(raw.time) = thresh.month AND DAY(raw.time) = thresh.day
    WHERE ROUND(raw.latitude, 4) = ROUND(40.708282, 4) 
        AND ROUND(raw.longitude, 4) = ROUND(5.446646, 4)
        AND raw.time BETWEEN '2016-01-01' AND '2016-02-15'
    ORDER BY raw.time
""").df()
print(check_temps)

         time  sst_celsius  p90_celsius
0  2016-01-01    16.969994    16.239994
1  2016-01-02    16.949994    16.159994
2  2016-01-03    16.879994    16.059994
3  2016-01-04    16.679994    16.009994
4  2016-01-05    16.249994    15.939994
5  2016-01-06    16.169994    15.849994
6  2016-01-07    16.179994    15.839994
7  2016-01-08    16.079994    15.809994
8  2016-01-09    16.539994    15.709994
9  2016-01-10    16.509994    15.659994
10 2016-01-11    16.869994    15.589994
11 2016-01-12    16.399994    15.529994
12 2016-01-13    16.659994    15.459994
13 2016-01-14    15.959994    15.399994
14 2016-01-15    16.139994    15.319994
15 2016-01-16    15.849994    15.179994
16 2016-01-17    15.429994    15.119994
17 2016-01-18    15.379994    15.149994
18 2016-01-19    15.229994    15.079994
19 2016-01-20    15.649994    14.989994
20 2016-01-21    15.829994    14.989994
21 2016-01-22    15.539994    14.949994
22 2016-01-23    15.509994    14.969994
23 2016-01-24    15.269994    14.979994


**Investigation note**: the first detected event initially looked suspicious, a 46-day run (Jan 1 - Feb 15, 2016) at the very start of the dataset, during the coldest months of the year. Checked the raw SST against the threshold for that cell and period: every single day is genuinely above its calendar-day p90 threshold, by 0.09-1.28 C, consistently across the full 46 days. Not a filtering bug.

Marine heatwaves are not exclusively a summer phenomenon: the Hobday definition flags anomalies relative to the historical norm for that specific calendar day, not an absolute temperature. A persistently mild winter is a legitimate winter heatwave under this methodology, similar in kind to real documented events like the 2014-2016 Northeast Pacific "Blob", which spanned multiple seasons.

One honest caveat worth flagging: 2016 is the first year of the 11-year baseline used here (a documented simplification of the standard 30-year climatological baseline used in Hobday et al., reduced here to 11 years for project scope). With a shorter baseline, an unusually mild winter early in the series carries proportionally more weight on its own threshold than it would with 30 years of data. This does not invalidate the detected event, but is noted as a limitation of the shortened baseline.

In [21]:
# Basic statistics on the detected events
print(result_events["duration_days"].describe())

count    253145.000000
mean         12.348140
std          11.650555
min           5.000000
25%           6.000000
50%           8.000000
75%          13.000000
max         161.000000
Name: duration_days, dtype: float64


In [22]:
# Investigate the longest detected event
longest_event = result_events[result_events["duration_days"] == 161]
print(longest_event)

         latitude  longitude  group_id event_start  event_end  duration_days
173143  40.507652  13.904411      2110  2024-10-22 2025-03-31            161
173148  40.557808  13.904411      2082  2024-10-22 2025-03-31            161
173157  40.507652  13.704227      2164  2024-10-22 2025-03-31            161
173164  40.507652  13.954456      2087  2024-10-22 2025-03-31            161


In [23]:
check_longest = con.execute("""
    SELECT raw.time, raw.analysed_sst - 273.15 AS sst_celsius, thresh.p90_celsius
    FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
    INNER JOIN '../data/processed/daily_thresholds.parquet' AS thresh
        ON MONTH(raw.time) = thresh.month AND DAY(raw.time) = thresh.day
    WHERE ROUND(raw.latitude, 4) = ROUND(40.507652, 4) 
        AND ROUND(raw.longitude, 4) = ROUND(13.954456, 4)
        AND raw.time BETWEEN '2024-10-22' AND '2025-03-31'
    ORDER BY raw.time
""").df()
print(check_longest.describe())

                      time  sst_celsius  p90_celsius
count                  161   161.000000   161.000000
mean   2025-01-10 00:00:00    17.504838    16.753472
min    2024-10-22 00:00:00    14.809994    14.549994
25%    2024-12-01 00:00:00    15.589994    14.809994
50%    2025-01-10 00:00:00    16.359994    15.659994
75%    2025-02-19 00:00:00    18.969993    18.089993
max    2025-03-31 00:00:00    22.979993    22.479993
std                    NaN     2.431433     2.435070


**Investigation note**: the longest detected event (161 days, from October 22nd 2024 to March 31st 2025) is near the Gulf of Naples, at the southeastern edge of the bounding box, close to areas flagged by the Mare Caldo project for ecological stress. Checked actual SST against threshold across the full period: consistently 0.7-0.8 C above threshold on average (mean 17.50 C observed vs 16.75 C threshold), consistent with a systematic gap (by construction, every flagged day exceeds its threshold; column-level quantiles shown here for a quick visual sense of magnitude, not as proof of per-day consistency), not driven by a single spike. This is consistent with real documented Mediterranean heatwave activity in 2024-2025 (up to 30.8 C off the Cote d'Azur in August 2024, per earlier research). A second independently verified event, alongside the January-February 2016 case, adding confidence in the aggregate results (253,145 events, median duration 8 days) beyond trusting the raw count alone.

**Results**: 253,145 valid events detected across the full bounding box (2016-2026), one per consecutive run per grid cell. Duration statistics: median 8 days, mean 12.35 days (pulled upward by a long right tail, some events last months), minimum 5 days (the Hobday cutoff), maximum 161 days. Both investigated events, the 2016 case (46 days) and the longest overall (161 days), were individually verified against raw SST data and confirmed as genuine sustained anomalies, not artifacts (see investigation notes above).